In [4]:
# =======================
# PRUEBA COMPLETA (mínima)
# =======================

import time
import random
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup
from collections import namedtuple

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9"
}

# -------------------
# get_lat_lon (tu versión)
# -------------------
def get_lat_lon(url, headers):
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    s = soup.find("script", id="__PRELOADED_STATE__")
    if not s:
        return None, None

    raw = s.get_text(strip=False)

    # recortamos al JSON grande
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not m:
        return None, None

    data = json.loads(m.group(0))

    # 🔎 búsqueda DIRIGIDA de map_info → location
    def find_map_info(obj):
        if isinstance(obj, dict):
            if "map_info" in obj:
                loc = obj["map_info"].get("location", {})
                lat = loc.get("latitude")
                lon = loc.get("longitude")
                if lat is not None and lon is not None:
                    return lat, lon
            for v in obj.values():
                res = find_map_info(v)
                if res != (None, None):
                    return res

        elif isinstance(obj, list):
            for it in obj:
                res = find_map_info(it)
                if res != (None, None):
                    return res

        return None, None

    return find_map_info(data)

# -------------------
# get_desc (para prueba)
# -------------------
def get_desc(url, headers):
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    m = soup.select_one("meta[property='og:description']")
    if m and m.get("content"):
        desc = m["content"].strip()
        if desc:
            return desc

    m = soup.select_one("meta[name='twitter:description']")
    if m and m.get("content"):
        desc = m["content"].strip()
        if desc:
            return desc

    p = soup.select_one("p.ui-pdp-description__content")
    if p:
        desc = p.get_text(" ", strip=True)
        if desc:
            return desc

    cont = soup.select_one("#description")
    if cont:
        p2 = cont.find("p")
        if p2:
            desc = p2.get_text(" ", strip=True)
            if desc:
                return desc

    return None

# -------------------
# parse_item mínimo (solo link + campos básicos)
# Ajusta/expande si ya tienes precio/baños/etc.
# -------------------
Vivienda = namedtuple("Vivienda", ["titulo", "link"])

def parse_item(li):
    titulo = None
    t = li.select_one("h2.ui-search-item__title") \
        or li.select_one("h2.poly-component__title-wrapper") \
        or li.select_one("h2")
    if t:
        titulo = t.get_text(strip=True)

    a = li.find("a", href=True)
    link = a["href"] if a else None

    return Vivienda(titulo, link)

# -------------------
# scrap_page con coords + desc (para prueba)
# -------------------
def scrap_page(PAGINAS, URL, HEADERS, espera_s=2, espera_item=(2, 4)):
    articulos = []

    for page in range(1, PAGINAS + 1):
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"
        print(f"[INFO] page {page}: {url_pag}")

        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                art = parse_item(li)
                row = art._asdict()

                lat, lon, desc = (None, None, None)
                if row.get("link"):
                    try:
                        clean_link = row["link"].split("#")[0]
                        lat, lon = get_lat_lon(clean_link, HEADERS)
                    except Exception as e:
                        print(f"[WARN] lat/lon: {e}")

                    try:
                        desc = get_desc(row["link"], HEADERS)
                    except Exception as e:
                        print(f"[WARN] desc: {e}")

                    time.sleep(random.uniform(*espera_item))  # pausa entre anuncios

                row["latitud"] = lat
                row["longitud"] = lon
                row["descripcion"] = desc

                articulos.append(row)

            except Exception as exc:
                print(f"[WARN] item con error: {exc}")

        time.sleep(espera_s)  # pausa entre páginas

    return pd.DataFrame(articulos)



In [5]:
# =======================
# EJECUTA UNA PRUEBA
# =======================

URL_TEST = "https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/"

df_test = scrap_page(
    PAGINAS=1,
    URL=URL_TEST,
    HEADERS=HEADERS,
    espera_s=2,          # pausa entre páginas
    espera_item=(3, 6)   # pausa entre anuncios (sube a (10,30) cuando confirmes que sirve)
)

df_test.head(5)

[INFO] page 1: https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/
[WARN] No se hallaron items en página 1.


""


In [3]:
df_test

,titulo,link,latitud,longitud,descripcion
0,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,[DZD-168]Departamento en División del Norte en...
1,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,¡PROYECTO EN PREVENTA! *Programa de Vivienda e...
2,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,[DCW-115]Excelentes Preventas en Ajusco Coyoac...
3,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,[DCQ-059]Exclusivo Departamento en Residencial...
4,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,[DCK-094]Hermoso departamento ubicado dentro d...
5,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,¡Vive la experiencia de estrenar tu departamen...
6,None,https://departamento.mercadolibre.com.mx/MLM-4...,None,None,Departamento en Venta en Los Reyes Coyoacán-DI...
7,None,https://departamento.mercadolibre.com.mx/MLM-4...,None,None,Venta Departamento en Coyoacán División del No...
8,None,https://departamento.mercadolibre.com.mx/MLM-4...,None,None,"Departamento en venta en Joy Coyoacán $ 5,950,..."
9,None,https://departamento.mercadolibre.com.mx/MLM-2...,None,None,"Loft en venta en el centro de Coyoacán, en una..."
